# PPO agents

> PPO based agent

In [ ]:
#| default_exp agents.rl.hyper

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

# set logging level to INFO
logging.basicConfig(level=logging.INFO)

from abc import ABC, abstractmethod
from typing import Union, Optional, List, Tuple
import numpy as np
import os
# file: ddopai/agents/hyper_predict.py
# ---------------------------------------------------------

import json
from pathlib import Path
from types import SimpleNamespace

from ddopai.envs.base import BaseEnvironment
from ddopai.agents.base import BaseAgent
from ddopai.utils import MDPInfo, Parameter
from ddopai.meta_learning.utils.helpers import get_latent_for_policy
import torch
import torch.optim as optim
import torch.nn.functional as F
from torchinfo import summary
import argparse, json, os, torch
import torch.nn.functional as F
import numpy as np
from types import SimpleNamespace    
import time

In [ ]:
#| export
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class HyperAgent(BaseAgent):          # extend your own BaseAgent if you have one
    """
    A *pure-inference* agent that uses the networks trained by Hyper.
    No gradients, no replay buffer – just (obs, a, r) -> action.
    """
    

    # -----------------------------------------------------------------
    # 1) Initialisation -- load everything from <run-dir>/models/*.pt
    # -----------------------------------------------------------------
    def __init__(self,
                 run_dir: str,                    # folder that contains models/
                 deterministic: bool = True,
                 force_device: str | None = None,
                 environment_info: Optional[MDPInfo] = None,
                 obsprocessors: list | None = None):   # "cpu"/"cuda"/None
        self.device = torch.device(force_device) if force_device else device
        mdl_dir     = os.path.join("models", run_dir)
        self.preprocessors = obsprocessors if obsprocessors is not None else []

        # --- load config so we still have the flags ------------------
        cfg_path = os.path.join(mdl_dir, "config.json")
        with open(cfg_path, "r") as f:
            self.args = argparse.Namespace(**json.load(f))

        # minimal fields that `Policy` helpers look at
        self.args.add_nonlinearity_to_latent = getattr(
            self.args, "add_nonlinearity_to_latent", False
        )
        self.args.pass_latent_to_policy = True

        # --- load networks ------------------------------------------
        self.policy  = torch.load(os.path.join(mdl_dir, "policy.pt"),
                                  map_location=self.device).eval()
        self.encoder = torch.load(os.path.join(mdl_dir, "encoder.pt"),
                                  map_location=self.device).eval()

        # placeholders that will be filled in start_episode ----------
        self.latent_sample = self.latent_mean = self.latent_logvar = None
        self.hidden_state  = None
        self.prev_obs      = None
        self.train_mode = "pretrained"  # "pretrained" or "finetuned"
        self.deterministic = deterministic
        super().__init__(environment_info = environment_info, obsprocessors = obsprocessors, agent_name = "hyper")
    # -----------------------------------------------------------------
    # 2) Called by the evaluation loop at the very start of every task
    # -----------------------------------------------------------------
    def start_episode(self, first_obs: np.ndarray):
        """
        Initialise the latent prior and internal RNN hidden state.
        Call this once right after env.reset().
        """
        with torch.no_grad():
            z_s, z_m, z_v, h = self.encoder.prior(batch_size=1, sample=True)

        self.latent_sample, self.latent_mean, self.latent_logvar = \
            z_s.to(self.device), z_m.to(self.device), z_v.to(self.device)
        self.hidden_state = h.to(self.device)

        self.prev_obs = torch.as_tensor(first_obs, dtype=torch.float32,
                                        device=self.device).unsqueeze(0)

    # -----------------------------------------------------------------
    # 3) Action selection
    # -----------------------------------------------------------------
    def draw_action_(self, obs: np.ndarray) -> np.ndarray:   # DDOP naming
        """
        One environment interaction step:   obs -> action
        """
        state_t = torch.as_tensor(obs, dtype=torch.float32,
                                  device=self.device).unsqueeze(0)

        latent = get_latent_for_policy(self.args,
                                        latent_sample=self.latent_sample,
                                        latent_mean=self.latent_mean,
                                        latent_logvar=self.latent_logvar)

        with torch.no_grad():
            _, action, _ = self.policy.act(state=state_t,
                                           latent=latent,
                                           belief=None, task=None,
                                           deterministic=self.deterministic)

        return action.squeeze(0).cpu().numpy()

    # -----------------------------------------------------------------
    # 4) Tell the encoder the transition we just saw
    # -----------------------------------------------------------------
    def observe(self,
                next_obs: np.ndarray,
                action:    np.ndarray,
                reward:    float,
                done:      bool):
        """
        Feed the (s, a, r, s') tuple into the RNN encoder so the latent
        posterior is up-to-date for the next step.
        """
        if done:                                                 # end of task
            return                                              # no update

        a_t = torch.as_tensor(action, dtype=torch.float32,
                              device=self.device).unsqueeze(0)
        r_t = torch.as_tensor([reward], dtype=torch.float32,
                              device=self.device).unsqueeze(0)
        s_next = torch.as_tensor(next_obs, dtype=torch.float32,
                                 device=self.device).unsqueeze(0)

        with torch.no_grad():
            z_s, z_m, z_v, h = self.encoder(actions=a_t,
                                            states=s_next,
                                            rewards=r_t,
                                            prev_states=self.prev_obs,
                                            hidden_state=self.hidden_state,
                                            return_prior=False,
                                            sample=True)

        self.latent_sample, self.latent_mean, self.latent_logvar = \
            z_s.unsqueeze(0), z_m.unsqueeze(0), z_v.unsqueeze(0)
        self.hidden_state = h
        self.prev_obs     = s_next


